# Baseline vs Version 1: tumor sizing on the same CT slice

Browse the same KiTS23 case/slice for both systems. This notebook is **CPU-only**:
it loads the saved text predictions, saved masks and prepared CT scans; it does not
run either model or change scores.

**Left:** baseline's reported major/minor endpoints (orange solid lines) and a
dashed ellipse reconstructed from them. **Right:** Version 1's saved predicted
mask (magenta) and the physical-space ellipse used for its measurement. Optional
green overlays show the target contour and reference ellipse.

The baseline emits text, not an ellipse object. Its endpoints can disagree about
the center, fail to be perpendicular, or disagree with the final numeric answer.
The dashed orange ellipse averages the two centers and uses the reported major
axis direction and both endpoint lengths: it is a **visual reconstruction**, not
an extra baseline prediction. The original segments remain visible and the
inconsistencies are reported below the figure. Unparseable endpoints are never
filled in from ground truth or the final lengths.

These are official **test** examples: use this notebook for descriptive comparison,
not to choose dropout, learning rate, or training duration.


## Paths and dependencies

Run from the repository or `notebooks/`. Defaults use the locally prepared scans
and the evaluation files copied from ICE. Adjust the three data paths below for
another machine. You need `requirements-cpu.txt` plus `IPython`; `ipywidgets` is
optional for the interactive browser. A Jupyter kernel normally includes IPython.

If needed, after running the path cell, install into this notebook's kernel with:
`%pip install -r "{ROOT / 'requirements-cpu.txt'}" ipywidgets`

No GPU, Torch, Transformers, Hugging Face downloads or weights are required.


In [ ]:
from pathlib import Path
import sys
import json

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / 'medmeasure').is_dir() and (p / 'configs/kits23-pilot.json').is_file()), None)
if ROOT is None:
    raise FileNotFoundError('Open this notebook from the MedMeasure repository.')
sys.path.insert(0, str(ROOT))

DATASET_DIR = ROOT / 'data/medvision/Datasets/KiTS23'
RUN_DIR = ROOT / 'results/v1-eval-6252813'
PAIRED_FILE = ROOT / 'results/v1-baseline-paired-6252813/paired.jsonl'
EXPORT_DIR = ROOT / 'runs/baseline-v1-visual-review'

for path in [DATASET_DIR / 'Images', RUN_DIR / 'masks', RUN_DIR / 'selected_rows.jsonl', PAIRED_FILE]:
    if not path.exists():
        raise FileNotFoundError(f'Missing visualization input: {path}')
print('Repository:', ROOT)
print('Native scans:', DATASET_DIR)
print('Saved Version 1 outputs:', RUN_DIR)


## Load the paired outputs

The paired file was generated by `scripts/compare_version1_baseline.py` from all
32 baseline shard logs and the Version 1 predictions. Its row identities,
manifest hash and scored references were checked against baseline provenance.
To regenerate it, use the command in `results/v1-baseline-paired-6252813/README.md`
or invoke that script with `--version1-run` and a new `--output` directory.

The viewer resolves masks under `RUN_DIR`, so saved absolute Mac paths in the
paired table do not prevent relocation. Native grid/header checks and mask-derived
ellipse measurements are checked when each slice is opened.


In [ ]:
# Render inside Jupyter; avoid native macOS plotting windows.
get_ipython().run_line_magic("matplotlib", "inline")

import importlib
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
# Reload the local helper when rerunning an already-open notebook.
import medmeasure.comparison_viz as comparison_viz
importlib.reload(comparison_viz)
from medmeasure.comparison_viz import ComparisonViewer, baseline_axes

viewer = ComparisonViewer(DATASET_DIR, RUN_DIR, PAIRED_FILE)
case_count = len({r['case_id'] for r in viewer.pairs})
print(f'Loaded {len(viewer.pairs):,} paired slices across {case_count} cases.')
print('Baseline coordinates: lower-left origin; displayed row = (1 - y) × height.')
DEFAULT_SAMPLE = next(r['sample_id'] for r in viewer.pairs if r['version1_status'] == 'success')


## View one example

Change `SAMPLE_ID`, or use `show_case(case_id, slice_idx)` below. The full field of
view is the default. Target-centered zoom is optional and can clip distant wrong
predictions; both panels always use the same limits.

The displayed baseline final answer is what was scored by the official parser.
The endpoint-derived lengths are displayed separately, exposing possible
localization/arithmetic inconsistencies. Neither changes the saved benchmark scores.


In [ ]:
SAMPLE_ID = DEFAULT_SAMPLE
_ = viewer.show(SAMPLE_ID, show_reference=True, zoom=False)


## Interactive case/slice browser

Select a case and slice. Toggle ground truth off for an uncluttered comparison or
zoom to the target. Reference overlays are review aids and were not model inputs.
If widgets are unavailable, use the manual cell below instead.


In [ ]:
try:
    controls = viewer.browse()
except ModuleNotFoundError as error:
    if error.name != 'ipywidgets':
        raise
    print('Optional browser needs ipywidgets. Use show_case() below, or install ipywidgets in this kernel.')


## Manual selection and difficult examples

List failures, cases where the baseline has lower relative error, or cases where
Version 1 has lower relative error. The comparison table keeps all failures;
both-successful error comparisons use the same rows. These are descriptive test
examples, not a validation tuning set.


In [ ]:
def show_case(case_id, slice_idx, show_reference=True, zoom=False):
    sample = next(r['sample_id'] for r in viewer.pairs
                  if r['case_id'] == case_id and r['slice_idx'] == slice_idx)
    return viewer.show(sample, show_reference=show_reference, zoom=zoom)

failures = [r for r in viewer.pairs if r['version1_status'] != 'success']
baseline_wins = [r for r in viewer.pairs if r['version1_status'] == 'success'
                 and r['baseline_success'] and r['baseline_mre'] < r['version1_mre']]
version1_wins = [r for r in viewer.pairs if r['version1_status'] == 'success'
                 and r['baseline_success'] and r['version1_mre'] < r['baseline_mre']]
for name, group in [('Version 1 failures', failures), ('Baseline lower MRE', baseline_wins),
                    ('Version 1 lower MRE', version1_wins)]:
    print(f'{name}: {len(group)} rows. First five:')
    print([(r['case_id'], r['slice_idx'], r['version1_status']) for r in group[:5]])
# Example: show_case('case_00145', 1)  # empty-mask failure; no V1 ellipse fabricated


## Inspect the original baseline response

This includes both its claimed endpoints and the final lengths. Check the response
when a reconstructed ellipse looks surprising or the endpoint-derived lengths
and final answer disagree.


In [ ]:
pair = viewer.by_id[getattr(viewer, 'current_sample_id', SAMPLE_ID)]
print(pair['baseline_response'])


## Export a side-by-side figure

The filename uses the last displayed sample (including the interactive browser). Exports are local review figures and do not modify predictions or scores.


In [ ]:
EXPORT_DIR.mkdir(parents=True, exist_ok=True)
EXPORT_SAMPLE = getattr(viewer, 'current_sample_id', SAMPLE_ID)
figure, endpoint_details = viewer.figure(EXPORT_SAMPLE, show_reference=True, zoom=False)
export_path = EXPORT_DIR / (EXPORT_SAMPLE.replace(':', '_') + '.png')
figure.savefig(export_path, dpi=150, facecolor='white')
plt.close(figure)
print('Saved:', export_path)


## Coordinate and geometry provenance

- The renderer is pinned to MedVision commit `45fa8e52684a65da6e1c0ef6ef422d5d4c3e94bd`:
  [medvision_utils.py](https://github.com/YongchengYAO/MedVision/blob/45fa8e52684a65da6e1c0ef6ef422d5d4c3e94bd/src/medvision_bm/medvision_lmms_eval/lmms_eval/tasks/medvision/medvision_utils.py).
  Its SHA256 matches the pilot pin: `d65b554e16a00a0dd76a490321ba6b79d90d46d834668ae6aea60ffd89ca0ddf`.
- `_load_nifti_2d` uses `volume[:, :, slice_idx]` for axial images; `doc_to_visual`
  calls `Image.fromarray` directly. No transpose, flip, crop or padding is applied.
  Resizing preserves the whole frame. Endpoint coordinates independently use a
  **lower-left origin**, verified in the pinned
  [sizing training target code](https://github.com/YongchengYAO/MedVision/blob/45fa8e52684a65da6e1c0ef6ef422d5d4c3e94bd/src/medvision_bm/sft/sft_utils.py#L1336).
  Native column = x × width; native row = (1 − y) × height.
  This corrects the original viewer's vertically mirrored baseline overlays;
  saved sizing predictions and scores are unaffected.
- Both panels use the same native CT rendering with HU window [-160, 240], not a
  claim of identical rendered inputs to both models. The official baseline has its
  own prompt/rendering/resolution; its prompt can state a 504×504 processor canvas.
- Ellipses are sampled in physical millimeters and then divided by column/row
  spacing independently. This preserves anisotropic geometry and avoids rotating
  an already rescaled ellipse incorrectly.
- Version 1's ellipse is recomputed from the saved binary mask using the same
  `fit_mask` implementation as scoring and checked against saved axis lengths.
  An empty mask or failed fit is shown as a failure, without an invented ellipse.
- A green reference ellipse is shown only if a ground-truth component fit matches
  the full-precision planner axes. Otherwise only the target contour is drawn.
- Final lengths alone do not determine an ellipse's center or orientation. For
  six of the 3,179 baseline responses, the strict endpoint parser cannot recover
  two unambiguous coordinate pairs per axis; the notebook shows no baseline ellipse
  for those rows (the answer and scored errors remain available).
